# Universal Multi-Crop & Multi-Region Yield Prediction Engine

This notebook demonstrates the **universal multi-crop and multi-region machine learning pipeline** for agricultural crop yield prediction in tonnes per hectare (t/ha) across Maharashtra, India.

### Supported Trained Crops:
- **Soybean** (*Glycine max*)
- **Cotton** (*Gossypium hirsutum*)
- **Groundnut** (*Arachis hypogaea*)
- **Maize** (*Zea mays*)
- **Wheat** (*Triticum aestivum*)
- **Chickpea / Pigeonpea** (*Cicer arietinum* / *Cajanus cajan*)
- **Rice / Sorghum** (*Oryza sativa* / *Sorghum bicolor*)

### Validated Agroclimatic Districts:
Ahmednagar, Akola, Amravati, Aurangabad, Beed, Bhandara, Buldhana.


In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configure aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)

# Locate multi-crop dataset
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
csv_path = project_root / 'data' / 'processed' / 'crop_yield_multicrop.csv'
df = pd.read_csv(csv_path)
print(f"Loaded multi-crop dataset: {df.shape[0]} rows across {df['crop_id'].nunique()} crops.")
print(df['crop_id'].value_counts())

## 1. Multi-Crop Yield Distributions by Crop Variety
Examining historical yield variations across all trained crops in Maharashtra.

In [ ]:
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='crop_id', y='yield_t_ha', palette='Set2')
plt.title('Historical Crop Yield Distributions across Maharashtra Districts (t/ha)', fontsize=14, fontweight='bold')
plt.xlabel('Crop Variety', fontsize=12)
plt.ylabel('Yield (tonnes / hectare)', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

## 2. Load the Production Shared Multi-Crop Random Forest Model
The production model pools cross-crop agroclimatic relationships (R² = 0.509 on spatial cross-validation).

In [ ]:
model_path = project_root / 'backend' / 'models' / 'experiments' / 'multicrop' / 'shared_rf' / 'model.joblib'
model = joblib.load(model_path)
print(f'Model Architecture: {type(model).__name__}')
print(f'Pipeline Steps: {[name for name, _ in model.steps]}')
regressor = model.named_steps['regressor']
print(f'Random Forest Estimators: {regressor.n_estimators}, Max Depth: {regressor.max_depth}')

## 3. Predict Yield for Multiple Crops Under Same Weather/Soil Conditions
Demonstrating multi-crop inference using the exact same shared Random Forest model.

In [ ]:
# Construct sample observations for different crops in Akola district
test_crops = ['soybean', 'cotton', 'groundnut', 'maize', 'wheat']
sample_records = []
for crop in test_crops:
    sample_records.append({
        'crop_id': crop,
        'district': 'Akola',
        'year': 2024,
        'soil_ph': 7.4,
        'organic_carbon_pct': 0.65,
        'clay_pct': 45.0,
        'sand_pct': 20.0,
        'silt_pct': 35.0,
        'bulk_density_g_cm3': 1.25,
        'cation_exchange_capacity_cmol_kg': 38.0,
        'cumulative_rainfall_mm': 850.5 if crop != 'wheat' else 45.0,
        'mean_temp_max_c': 31.8 if crop != 'wheat' else 28.5,
        'mean_temp_min_c': 22.4 if crop != 'wheat' else 14.0,
        'mean_solar_radiation_mj_m2': 18.6 if crop != 'wheat' else 19.5,
        'mean_relative_humidity_pct': 76.2 if crop != 'wheat' else 48.0,
        'consecutive_dry_days_max': 8 if crop != 'wheat' else 35,
        'heat_stress_days': 4 if crop != 'wheat' else 0
    })

sample_df = pd.DataFrame(sample_records)
preds = model.predict(sample_df)

results_df = pd.DataFrame({
    'Crop': [c.capitalize() for c in test_crops],
    'District': ['Akola'] * len(test_crops),
    'Predicted Yield (t/ha)': [round(p, 3) for p in preds],
    'Predicted Yield (qt/acre)': [round(p * 4.047, 2) for p in preds],
    'Predicted Yield (kg/ha)': [round(p * 1000, 1) for p in preds]
})
display(results_df)

## 4. Multi-Crop Feature Importance Breakdown
Top features driving the Shared Multi-Crop Random Forest decisions.

In [ ]:
fi_csv_path = project_root / 'reports' / 'model_analysis' / 'top10_feature_importance.csv'
if fi_csv_path.exists():
    fi_df = pd.read_csv(fi_csv_path)
    display(fi_df)
    
    plt.figure(figsize=(10, 5))
    sns.barplot(data=fi_df, x='importance', y='feature', palette='viridis')
    plt.title('Top-10 Feature Importances in Shared Multi-Crop Random Forest', fontsize=13, fontweight='bold')
    plt.xlabel('Impurity-based Gini Importance')
    plt.ylabel('Agroclimatic & Soil Feature')
    plt.tight_layout()
    plt.show()